In [ ]:
# =====================================================================
# Анализ кода: Complement Naive Bayes для классификации спама с визуализацией
# =====================================================================

# Импорт необходимых библиотек
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Библиотеки для обработки текста и построения TF-IDF признаков
from sklearn.feature_extraction.text import TfidfVectorizer

# Импорт модели Комплементарного наивного Байеса (ComplementNB)
from sklearn.naive_bayes import ComplementNB

# Импорт функций для разделения данных и оценки модели
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc

# Импорт NMF для уменьшения размерности (сохранение неотрицательности)
from sklearn.decomposition import NMF
from matplotlib.colors import ListedColormap
import seaborn as sns

# =====================================================================================
# 1. Загрузка и предобработка данных
# =====================================================================================
# Загружаем датасет "spam.csv", который содержит информацию о спаме и не спаме.
# В данном случае используются только колонки "Category" (метка, например, 'ham' или 'spam')
# и "Message" (текст сообщения).
df = pd.read_csv("../data/spam.csv", encoding="latin-1")
# Оставляем только нужные колонки для анализа
df = df[['Category', 'Message']]

# Преобразуем метки: 'ham' → 0 (не спам), 'spam' → 1 (спам)
df['Label'] = df['Category'].map({'ham': 0, 'spam': 1})

# Выводим первые строки датасета для проверки корректности загрузки и преобразования
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 100)
print(df.head())
print(df.info())
print(df.describe())



# =====================================================================================
# 2. Векторизация текста
# =====================================================================================
# Преобразуем текст сообщений в числовые признаки с помощью TF-IDF векторизации.
# TF-IDF (Term Frequency - Inverse Document Frequency) позволяет оценить значимость каждого слова
# в документе относительно всего корпуса, исключая слишком часто встречающиеся слова.
vectorizer = TfidfVectorizer(stop_words='english')
X_features = vectorizer.fit_transform(df['Message'])  # Получаем разреженную матрицу признаков
y = df['Label'].values  # Целевая переменная: 0 – не спам, 1 – спам

# =====================================================================================
# 3. Разделение на обучающую и тестовую выборки
# =====================================================================================
# Делим данные на обучающую и тестовую выборки в соотношении 70/30.
# random_state=42 обеспечивает воспроизводимость разбиения.
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.3, random_state=42)

# =====================================================================================
# 4. Обучение модели ComplementNB
# =====================================================================================
# Инициализируем модель Комплементарного наивного Байеса, которая хорошо работает с разреженными данными,
# характерными для текстовой классификации.
clf = ComplementNB(alpha=1.0)
clf.fit(X_train, y_train)

# Получаем предсказания модели на тестовой выборке
y_pred = clf.predict(X_test)

# Оценка результатов модели:
# - accuracy – общая точность модели;
# - classification_report – подробные метрики для каждого класса (precision, recall, f1-score);
# - confusion_matrix – матрица ошибок, показывающая, сколько примеров каждого класса было классифицировано верно или с ошибкой.
acc = accuracy_score(y_test, y_pred)
print("\nРезультаты модели на тестовой выборке:")
print("Accuracy:", acc)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# =====================================================================================
# 5. Визуализация результатов с помощью NMF (уменьшение размерности)
# =====================================================================================
# TF-IDF матрица имеет очень высокую размерность. Для визуализации результатов применяем NMF (Non-negative Matrix Factorization),
# которая позволяет уменьшить размерность данных до 2-х компонент, сохраняя неотрицательные значения.
# Это полезно для построения графиков с решающими границами.
# Преобразуем разреженные матрицы в плотный формат (numpy array)
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

# Применяем NMF для уменьшения размерности до 2-х компонент
nmf = NMF(n_components=2, random_state=42)
X_train_nmf = nmf.fit_transform(X_train_dense)
X_test_nmf = nmf.transform(X_test_dense)

# Обучаем новый экземпляр модели ComplementNB на данных, преобразованных с помощью NMF.
# Это делается для визуализации в двумерном пространстве.
clf_nmf = ComplementNB(alpha=1.0)
clf_nmf.fit(X_train_nmf, y_train)

# Функция для построения решающих границ в двумерном пространстве
def plot_decision_boundary(clf, X, y, title, step=0.01):
    # Определяем диапазоны для каждой компоненты с небольшим запасом по краям
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, step),
                         np.arange(y_min, y_max, step))

    # Предсказание классов для каждой точки сетки
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Строим контурную заливку, которая показывает области, отнесённые к разным классам
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(('lightgreen', 'salmon')))
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())

    # Отображаем точки исходного набора данных, раскрашенные по классам
    for i, j in enumerate(np.unique(y)):
        plt.scatter(X[y == j, 0], X[y == j, 1],
                    color=ListedColormap(('green', 'red'))(i), label=f"Class {j}", edgecolor='k')
    plt.title(title)
    plt.xlabel('Компонента 1')
    plt.ylabel('Компонента 2')
    plt.legend()
    plt.show()

# Визуализация на обучающем наборе в пространстве, полученном с помощью NMF
plot_decision_boundary(clf_nmf, X_train_nmf, y_train, title="ComplementNB - Обучающая выборка (NMF)")

# Визуализация на тестовой выборке в пространстве, полученном с помощью NMF
plot_decision_boundary(clf_nmf, X_test_nmf, y_test, title="ComplementNB - Тестовая выборка (NMF)")


acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)
print("Classification Report:\n", classification_report(y_test, y_pred))

# Матрица ошибок
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title('Матрица ошибок')
plt.show()

# ROC-кривая
y_proba = clf.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC-кривая (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('Ложно-положительные')
plt.ylabel('Истинно-положительные')
plt.title('ROC-кривая для ComplementNB')
plt.legend(loc='lower right')
plt.show()

# =====================================================================================
# Итоговый анализ и комментарии
# =====================================================================================
# 1. Загрузка и предобработка данных:
#    - Датасет "spam.csv" загружается с использованием корректной кодировки (latin-1).
#    - Из датасета выбираются только релевантные колонки ("Category" и "Message").
#    - Метки преобразуются: 'ham' становится 0 (не спам), 'spam' становится 1 (спам).
#
# 2. Векторизация текста:
#    - Использование TfidfVectorizer позволяет преобразовать текст в числовые признаки,
#      где учитывается частота появления терминов и их значимость.
#
# 3. Разделение данных:
#    - Данные разделяются на обучающую и тестовую выборки (70/30), что позволяет оценить способность модели к обобщению.
#
# 4. Обучение модели ComplementNB:
#    - Модель Complement Naive Bayes обучается на TF-IDF признаках.
#    - Оценка модели показывает:
#         • Accuracy ≈ 93.18%,
#         • Классификационный отчет демонстрирует высокие метрики для класса 0 (не спам) и несколько более низкие для класса 1 (спам),
#         • Матрица ошибок указывает на наличие ложных срабатываний, особенно для не спама (например, 107 примеров класса 0 ошибочно отнесены к спаму).
#
# 5. Визуализация с помощью NMF:
#    - Так как исходное пространство TF-IDF имеет высокую размерность, применяется NMF для сокращения размерности до 2-х,
#      что позволяет построить наглядные графики с решающими границами.
#    - Функция plot_decision_boundary строит контурную заливку, отображающую области, отнесённые к разным классам,
#      и накладывает на неё точки обучающей и тестовой выборок.
#
# 6. Выводы:
#    - Данный код демонстрирует полный цикл обработки текстовых данных для классификации спама:
#         • от загрузки и предобработки,
#         • через векторизацию и обучение модели,
#         • до визуализации результатов с помощью уменьшения размерности.
#    - Полученные метрики показывают, что модель эффективно определяет не спам, хотя для спама (класс 1) наблюдается
#      несколько меньшая точность, что может быть связано с дисбалансом классов.
#    - Визуализация решающих границ в двумерном пространстве (NMF) помогает оценить разделимость классов и понять, как
#      модель различает спам и не спам, что является полезным дополнением к количественной оценке модели.
#
# Данный подход демонстрирует, как можно применять алгоритмы машинного обучения для задач обработки естественного языка,
# а также как визуализировать результаты для лучшего понимания работы модели.
